# 3.3 Properties of Determinants

**Course:** Elementary Linear Algebra (Larson, 8e) 
**Chapter 3 — Determinants**

---

## 1. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '../..'))

In [ ]:
from linalg_core.matrix import Matrix
from linalg_core.ops import add, scalar_mul, multiply, transpose, inverse
from linalg_core.determinant import det_cofactor, det_elimination
from linalg_core.elimination import to_rref, solve
from linalg_core import EPSILON
import random

random.seed(42)


def random_matrix(n):
    """Generate a random n×n matrix with entries in [-10, 10]."""
    data = [random.uniform(-10, 10) for _ in range(n * n)]
    return Matrix(n, n, data)


def approx_equal(a, b, tol=1e-6):
    """Check if two scalars are approximately equal."""
    return abs(a - b) < tol


def matrices_equal(A, B):
    """Check if two matrices are equal within EPSILON tolerance."""
    if A.rows != B.rows or A.cols != B.cols:
        return False
    return all(abs(a - b) < EPSILON for a, b in zip(A.data, B.data))


print("Setup complete.")

---

## 2. Motivation

We have two algorithms for computing determinants: cofactor expansion (Section 3.1) and
elimination (Section 3.2). Now we ask the deeper question:

**What properties does $\det$ satisfy, and what does $\det(A) = 0$ really tell us?**

The determinant turns out to be far more than a number attached to a matrix. It is the
single scalar that separates the **invertible** world from the **singular** world. When
$\det(A) \neq 0$, systems have unique solutions, inverses exist, and columns are independent.
When $\det(A) = 0$, everything collapses.

Along the way, we will uncover a common student trap: the belief that $\det(cA) = c \cdot \det(A)$.
The correct formula involves $c^n$, and getting this wrong leads to spectacular errors.

Let's build the theory one property at a time.

---

## 3. Build — Core Properties

We develop five key ideas, each demonstrated computationally.

### 3.1 Scalar Multiples: $\det(cA) = c^n \det(A)$

**Theorem 3.6 (Larson).** If $A$ is an $n \times n$ matrix and $c$ is a scalar, then

$$\det(cA) = c^n \, \det(A).$$

**Why $c^n$?** Multiplying $A$ by $c$ scales *every row* by $c$. From Section 3.2, we know
that scaling one row multiplies the determinant by $c$. There are $n$ rows, so the total
factor is $c \cdot c \cdots c = c^n$.

**Common mistake:** Students often write $\det(2A) = 2 \det(A)$. This is **wrong** for any
$n > 1$. The correct statement is $\det(2A) = 2^n \det(A)$.

**Example (from Larson, Example 2).** Let
$$A = \begin{bmatrix} 1 & -2 & 4 \\ 3 & 0 & 5 \\ -2 & -3 & 1 \end{bmatrix}$$
so that $10A$ has entries scaled by 10. Then $\det(10A) = 10^3 \cdot \det(A)$.

In [ ]:
A = Matrix.from_lists([
    [1, -2, 4],
    [3,  0, 5],
    [-2, -3, 1]
])

n = A.rows
c = 10

det_A = det_cofactor(A)
cA = scalar_mul(A, c)
det_cA = det_cofactor(cA)

print(f"A ="); print(A)
print(f"\ndet(A) = {det_A}")
print(f"\n{c}A ="); print(cA)
print(f"\ndet({c}A) = {det_cA}")
print(f"c^n * det(A) = {c}^{n} * {det_A} = {c**n * det_A}")
print(f"\ndet(cA) == c^n * det(A)? {approx_equal(det_cA, c**n * det_A)}")

print("\n" + "=" * 60)
print("COMMON MISTAKE DEMO")
print("=" * 60)
wrong_answer = c * det_A
right_answer = c**n * det_A
print(f"\nWRONG: det({c}A) = {c} * det(A) = {wrong_answer}")
print(f"RIGHT: det({c}A) = {c}^{n} * det(A) = {right_answer}")
print(f"Actual: det({c}A) = {det_cA}")
print(f"\nThe wrong answer is off by a factor of {c**(n-1)}!")

print("\n" + "=" * 60)
print("SCALING DEMO: Different values of n")
print("=" * 60)
for size in [2, 3, 4, 5]:
    M = random_matrix(size)
    det_M = det_elimination(M)
    det_2M = det_elimination(scalar_mul(M, 2))
    expected = (2 ** size) * det_M
    ok = approx_equal(det_2M, expected)
    print(f"  n={size}: det(2A) = {det_2M:12.4f}  |  2^{size} * det(A) = {expected:12.4f}  |  Match: {ok}")

### 3.2 Invertibility: $\det(A) \neq 0 \iff A$ is Invertible

**Theorem 3.7 (Larson).** A square matrix $A$ is invertible if and only if $\det(A) \neq 0$.

This is the most important single fact in the determinant chapter. It means the determinant
is a perfect **test for invertibility**: compute one number and you know whether an inverse exists.

**Corollary (Theorem 3.8).** If $A$ is invertible, then $\det(A^{-1}) = \dfrac{1}{\det(A)}$.

Let's test both directions:
1. A **nonsingular** (invertible) matrix should have $\det \neq 0$, and the inverse should exist.
2. A **singular** matrix should have $\det = 0$, and attempting to invert should fail.

**Example (from Larson, Example 3).**
$$A = \begin{bmatrix} 0 & 2 & -1 \\ 3 & -2 & 1 \\ 3 & 2 & -1 \end{bmatrix} \quad (\text{singular: } |A| = 0)$$
$$B = \begin{bmatrix} 0 & 2 & -1 \\ 3 & -2 & 1 \\ 3 & 2 & 1 \end{bmatrix} \quad (\text{nonsingular: } |B| = -12)$$

In [ ]:
A_singular = Matrix.from_lists([
    [0,  2, -1],
    [3, -2,  1],
    [3,  2, -1]
])

B_nonsingular = Matrix.from_lists([
    [0,  2, -1],
    [3, -2,  1],
    [3,  2,  1]
])

det_A = det_elimination(A_singular)
det_B = det_elimination(B_nonsingular)

print("--- Singular matrix ---")
print("A ="); print(A_singular)
print(f"det(A) = {det_A}")
print(f"A is invertible? {abs(det_A) > EPSILON}")

try:
    inv_A = inverse(A_singular)
    print("Inverse found (unexpected!)")
except ValueError as e:
    print(f"Inverse attempt: {e}")

print("\n--- Nonsingular matrix ---")
print("B ="); print(B_nonsingular)
print(f"det(B) = {det_B}")
print(f"B is invertible? {abs(det_B) > EPSILON}")

inv_B = inverse(B_nonsingular)
print(f"\nB^(-1) ="); print(inv_B)

det_inv_B = det_elimination(inv_B)
print(f"\ndet(B^(-1)) = {det_inv_B:.6f}")
print(f"1/det(B) = {1.0/det_B:.6f}")
print(f"det(B^(-1)) == 1/det(B)? {approx_equal(det_inv_B, 1.0/det_B)}")

print("\n--- Verification: B * B^(-1) = I ---")
product = multiply(B_nonsingular, inv_B)
I3 = Matrix.identity(3)
print(f"B * B^(-1) ="); print(product)
print(f"Equals I? {matrices_equal(product, I3)}")

### 3.3 The Invertible Matrix Theorem (Growing List)

The most powerful result in introductory linear algebra is a theorem that collects
**many equivalent conditions** for invertibility. Each chapter adds more entries.
Here is our growing list so far:

**Invertible Matrix Theorem.** For an $n \times n$ matrix $A$, the following are equivalent:

| # | Condition | Chapter |
|---|---|---|
| 1 | $A$ is invertible | 2.3 |
| 2 | $\det(A) \neq 0$ | 3.3 (Thm 3.7) |
| 3 | $\text{RREF}(A) = I_n$ | 1.2 |
| 4 | $Ax = 0$ has only the trivial solution $x = 0$ | 1.1 |
| 5 | $A$ is a product of elementary matrices | 2.4 |

The remarkable fact is that **every one of these** is telling you the same thing from a
different angle. Let's verify all five computationally for a single matrix.

In [ ]:
A = Matrix.from_lists([
    [1, -2,  2],
    [0,  3,  2],
    [1,  0,  1]
])

print("A ="); print(A)
print("\n" + "=" * 60)
print("INVERTIBLE MATRIX THEOREM: Checking all 5 conditions")
print("=" * 60)

print("\n--- Condition 1: A is invertible ---")
try:
    A_inv = inverse(A)
    cond1 = True
    print(f"A^(-1) exists.")
    print(A_inv)
except ValueError:
    cond1 = False
    print("A is NOT invertible.")

print("\n--- Condition 2: det(A) != 0 ---")
d = det_elimination(A)
cond2 = abs(d) > EPSILON
print(f"det(A) = {d}")
print(f"det(A) != 0? {cond2}")

print("\n--- Condition 3: RREF(A) = I ---")
R = A.copy()
to_rref(R)
I3 = Matrix.identity(3)
cond3 = matrices_equal(R, I3)
print(f"RREF(A) ="); print(R)
print(f"RREF(A) == I? {cond3}")

print("\n--- Condition 4: Ax = 0 has only the trivial solution ---")
aug = Matrix(3, 4)
for i in range(3):
    for j in range(3):
        aug[i, j] = A[i, j]
    aug[i, 3] = 0.0
sol_type, result = solve(aug)
cond4 = (sol_type == "unique" and all(abs(x) < EPSILON for x in result))
print(f"Solution type: {sol_type}")
if sol_type == "unique":
    print(f"Solution: {result}")
print(f"Only trivial solution? {cond4}")

print("\n--- Condition 5: A is a product of elementary matrices ---")
print("(If RREF(A) = I, then the row operations that reduce A to I")
print("correspond to elementary matrices E_k...E_1 such that E_k...E_1 A = I,")
print("so A = E_1^(-1)...E_k^(-1), a product of elementary matrices.)")
cond5 = cond3
print(f"Product of elementary matrices? {cond5}")

print("\n" + "=" * 60)
print(f"All 5 conditions agree: {cond1 == cond2 == cond3 == cond4 == cond5}")
print(f"A is {'INVERTIBLE' if cond1 else 'SINGULAR'}")
print("=" * 60)

print("\n\n--- Now test with a SINGULAR matrix ---")
S = Matrix.from_lists([
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9]
])
print("S ="); print(S)

det_S = det_elimination(S)
print(f"\ndet(S) = {det_S}")
print(f"det(S) != 0? {abs(det_S) > EPSILON}")

R_S = S.copy()
to_rref(R_S)
print(f"\nRREF(S) ="); print(R_S)
print(f"RREF(S) == I? {matrices_equal(R_S, I3)}")

aug_S = Matrix(3, 4)
for i in range(3):
    for j in range(3):
        aug_S[i, j] = S[i, j]
    aug_S[i, 3] = 0.0
sol_type_S, result_S = solve(aug_S)
print(f"\nAx=0 solution type: {sol_type_S}")
if sol_type_S == "infinite":
    print("Nontrivial solutions exist (free variables present).")

try:
    inverse(S)
    print("\nInverse found (unexpected!)")
except ValueError as e:
    print(f"\nInverse attempt: {e}")

print("\nAll conditions consistently say: S is SINGULAR.")

### 3.4 Conditions for $\det(A) = 0$

We already know from Section 3.2 that certain row patterns force $\det(A) = 0$.
Let's collect them:

1. **Zero row (or column):** If any row of $A$ is all zeros, then $\det(A) = 0$.
2. **Two equal rows:** If rows $i$ and $j$ are identical, then $\det(A) = 0$.
3. **Proportional rows:** If row $i = k \cdot$ row $j$, then $\det(A) = 0$.

**Why?** 
- A zero row means $A$ cannot have full rank, so $A$ is singular.
- Two equal rows: swapping them should negate the determinant, but the matrix is unchanged.
  So $\det(A) = -\det(A)$, which forces $\det(A) = 0$.
- Proportional rows: subtract the multiple to create a zero row, which doesn't change the
  determinant (row addition property from Section 3.2).

All three are special cases of **linear dependence among rows**.

In [ ]:
print("=" * 60)
print("CONDITION 1: Zero row")
print("=" * 60)
A_zero_row = Matrix.from_lists([
    [1, 2, 3],
    [0, 0, 0],
    [4, 5, 6]
])
print("A ="); print(A_zero_row)
print(f"det(A) = {det_elimination(A_zero_row)}")

print("\n" + "=" * 60)
print("CONDITION 2: Two equal rows")
print("=" * 60)
A_equal_rows = Matrix.from_lists([
    [1, 2, 3],
    [4, 5, 6],
    [1, 2, 3]
])
print("A ="); print(A_equal_rows)
print(f"Row 1 == Row 3: {A_equal_rows.get_row(0) == A_equal_rows.get_row(2)}")
print(f"det(A) = {det_elimination(A_equal_rows)}")

print("\nWhy? Swapping rows 1 and 3 should negate det, but the matrix is unchanged.")
print("So det(A) = -det(A), which means det(A) = 0.")

print("\n" + "=" * 60)
print("CONDITION 3: Proportional rows")
print("=" * 60)
A_prop_rows = Matrix.from_lists([
    [1, 2, 3],
    [4, 5, 6],
    [2, 4, 6]
])
print("A ="); print(A_prop_rows)
print(f"Row 3 = 2 * Row 1: {[2*x for x in A_prop_rows.get_row(0)]}")
print(f"det(A) = {det_elimination(A_prop_rows)}")

print("\n" + "=" * 60)
print("LARGER EXAMPLE: 4x4 with proportional rows")
print("=" * 60)
A_4x4 = Matrix.from_lists([
    [1,  2,  3,  4],
    [5,  6,  7,  8],
    [9, 10, 11, 12],
    [3,  6,  9, 12]
])
print("A ="); print(A_4x4)
print(f"Row 4 = 3 * Row 1: {[3*x for x in A_4x4.get_row(0)]}")
print(f"det(A) = {det_elimination(A_4x4)}")

### 3.5 Determinants and Linear Independence (Preview)

The columns (or rows) of an $n \times n$ matrix $A$ are linearly independent **if and only if**
$\det(A) \neq 0$.

This connects determinants to Chapter 4's central theme (vector spaces, basis, dimension)
and gives us a **fast independence test**: just compute one number.

**First-principles reasoning.** Linear independence of the columns of $A$ means $Ax = 0$
has only the trivial solution. By the Invertible Matrix Theorem (Condition 4 above),
this is equivalent to $A$ being invertible, which is equivalent to $\det(A) \neq 0$.

Let's demonstrate: build a set of three $\mathbb{R}^3$ vectors, test independence via
determinant, then confirm by solving $Ax = 0$.

In [ ]:
print("=" * 60)
print("INDEPENDENT COLUMNS (det != 0)")
print("=" * 60)
v1 = [1, 2, 3]
v2 = [0, 1, 2]
v3 = [-2, 0, 1]

A_indep = Matrix.from_lists([
    [v1[0], v2[0], v3[0]],
    [v1[1], v2[1], v3[1]],
    [v1[2], v2[2], v3[2]]
])
print("Columns: v1 =", v1, ", v2 =", v2, ", v3 =", v3)
print("A (columns are v1, v2, v3) ="); print(A_indep)

det_indep = det_elimination(A_indep)
print(f"\ndet(A) = {det_indep}")
print(f"det != 0? {abs(det_indep) > EPSILON}")
print("Conclusion: columns are LINEARLY INDEPENDENT.")

aug_indep = Matrix(3, 4)
for i in range(3):
    for j in range(3):
        aug_indep[i, j] = A_indep[i, j]
    aug_indep[i, 3] = 0.0
sol_type, result = solve(aug_indep)
print(f"\nConfirmation via Ax=0: {sol_type}, x = {result}")

print("\n" + "=" * 60)
print("DEPENDENT COLUMNS (det = 0)")
print("=" * 60)
w1 = [1, 2, 3]
w2 = [4, 5, 6]
w3 = [5, 7, 9]
print(f"Columns: w1 = {w1}, w2 = {w2}, w3 = {w3}")
print(f"Note: w3 = w1 + w2 = {[a+b for a,b in zip(w1,w2)]}")

A_dep = Matrix.from_lists([
    [w1[0], w2[0], w3[0]],
    [w1[1], w2[1], w3[1]],
    [w1[2], w2[2], w3[2]]
])
print("A ="); print(A_dep)

det_dep = det_elimination(A_dep)
print(f"\ndet(A) = {det_dep}")
print(f"det == 0? {abs(det_dep) < EPSILON}")
print("Conclusion: columns are LINEARLY DEPENDENT.")

aug_dep = Matrix(3, 4)
for i in range(3):
    for j in range(3):
        aug_dep[i, j] = A_dep[i, j]
    aug_dep[i, 3] = 0.0
sol_type_dep, result_dep = solve(aug_dep)
print(f"\nConfirmation via Ax=0: {sol_type_dep}")
if sol_type_dep == "infinite":
    print("Nontrivial solutions exist -> dependent!")

---

## 4. Verify — Systematic Property Checks

Let's stress-test all the properties we've built with random matrices.

In [ ]:
print("=" * 60)
print("VERIFICATION 1: det(cA) = c^n * det(A)")
print("=" * 60)

random.seed(100)
all_pass = True
for trial in range(10):
    n = random.choice([2, 3, 4, 5])
    A = random_matrix(n)
    c = random.uniform(-5, 5)
    det_cA = det_elimination(scalar_mul(A, c))
    expected = (c ** n) * det_elimination(A)
    ok = approx_equal(det_cA, expected, tol=1e-3)
    if not ok:
        print(f"  Trial {trial+1} (n={n}, c={c:.2f}): FAILED  det(cA)={det_cA:.4f} vs c^n*det(A)={expected:.4f}")
        all_pass = False
    else:
        print(f"  Trial {trial+1} (n={n}, c={c:.2f}): PASSED")
print(f"\nOverall: {'ALL PASSED' if all_pass else 'SOME FAILED'}")

In [ ]:
print("=" * 60)
print("VERIFICATION 2: det = 0 conditions")
print("=" * 60)

print("\n--- Zero row ---")
random.seed(200)
for trial in range(5):
    A = random_matrix(4)
    zero_row = random.randint(0, 3)
    for j in range(4):
        A[zero_row, j] = 0.0
    d = det_elimination(A)
    print(f"  Trial {trial+1} (zero row {zero_row}): det = {d:.6f}, is zero? {abs(d) < EPSILON}")

print("\n--- Duplicate rows ---")
random.seed(300)
for trial in range(5):
    A = random_matrix(4)
    for j in range(4):
        A[3, j] = A[0, j]
    d = det_elimination(A)
    print(f"  Trial {trial+1} (row 4 = row 1): det = {d:.6f}, is zero? {abs(d) < EPSILON}")

print("\n--- Proportional rows ---")
random.seed(400)
for trial in range(5):
    A = random_matrix(4)
    k = random.uniform(-5, 5)
    for j in range(4):
        A[2, j] = k * A[1, j]
    d = det_elimination(A)
    print(f"  Trial {trial+1} (row 3 = {k:.2f}*row 2): det = {d:.6f}, is zero? {abs(d) < EPSILON}")

In [ ]:
print("=" * 60)
print("VERIFICATION 3: Invertible <=> det != 0")
print("=" * 60)

random.seed(500)
for trial in range(10):
    A = random_matrix(4)
    d = det_elimination(A)
    det_nonzero = abs(d) > EPSILON
    try:
        inv_A = inverse(A)
        invertible = True
    except ValueError:
        invertible = False
    consistent = (det_nonzero == invertible)
    print(f"  Trial {trial+1}: det={d:10.4f}, det!=0: {det_nonzero}, invertible: {invertible}, consistent: {consistent}")

---

## 5. Exercises

Test your understanding with the following problems. Write code in the provided cells.

### Exercise 1 (Easy)

Let $A$ be a $4 \times 4$ matrix with $\det(A) = 5$.

Without computing $\det(3A)$ directly, use the property $\det(cA) = c^n \det(A)$ to
find $\det(3A)$. Then verify your answer by constructing a concrete $4 \times 4$ matrix
with determinant 5 and computing $\det(3A)$ using `det_elimination`.

*Hint:* Start with $I_4$ and scale the $(1,1)$ entry to make $\det = 5$.

In [ ]:
# YOUR CODE HERE


### Exercise 2 (Medium)

Write a function `is_invertible(A)` that returns `True` if $A$ is invertible,
`False` otherwise. Use `det_elimination` as the test.

Then test your function on at least 5 matrices (a mix of invertible and singular).
For each matrix, print the matrix, `det(A)`, and the result of `is_invertible(A)`.

*Challenge extension:* Also verify by trying to compute the inverse and checking
whether your function's answer matches.

In [ ]:
# YOUR CODE HERE


### Exercise 3 (Challenge)

Write a function `verify_invertible_matrix_theorem(A)` that checks **all 5 conditions**
from the Invertible Matrix Theorem for a given $n \times n$ matrix $A$:

1. $A$ is invertible (try `inverse(A)`).
2. $\det(A) \neq 0$.
3. $\text{RREF}(A) = I_n$.
4. $Ax = 0$ has only the trivial solution.
5. (Implied by condition 3) $A$ is a product of elementary matrices.

The function should return a dictionary with the results and print a summary.
Then test it on 5 random $5 \times 5$ matrices.

*Key check:* All 5 conditions should **always agree** (all True or all False).
If they ever disagree, something is wrong.

In [ ]:
# YOUR CODE HERE


---

## Summary

| Property | Statement | Reference |
|---|---|---|
| Scalar multiple | $\det(cA) = c^n \det(A)$ for $n \times n$ | Theorem 3.6 |
| Product | $\det(AB) = \det(A) \det(B)$ | Theorem 3.5 |
| Invertibility test | $A$ invertible $\iff \det(A) \neq 0$ | Theorem 3.7 |
| Inverse determinant | $\det(A^{-1}) = 1/\det(A)$ | Theorem 3.8 |
| Transpose | $\det(A^T) = \det(A)$ | Theorem 3.9 |

**Conditions that force $\det(A) = 0$:**
- Zero row or column
- Two identical rows (or columns)
- Proportional rows (or columns)
- More generally: any linear dependence among rows or columns

**Invertible Matrix Theorem** (5 equivalent conditions so far):

$$A \text{ invertible} \iff \det(A) \neq 0 \iff \text{RREF}(A) = I \iff Ax=0 \text{ only trivial} \iff A = E_1 \cdots E_k$$

**Key takeaway:** The determinant is the *single number* that tells you whether a matrix
lives in the "nice" (invertible) world or the "degenerate" (singular) world. Every major
property of linear algebra splits along this dividing line.